# Day 2 · Session 3
# 03C ReAct and Guardrails

**Objective:** Understand ReAct loop and basic guardrails using simple Python.

## 1. Shared Tools
Run this cell first.

In [ ]:
import pandas as pd
import time

def calculator(expression: str):
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Calculation error: {e}"

def search_college_info(query: str):
    data = {
        "mtech_fee": "M.Tech CSE fee is Rs. 50,000 per semester. Duration is 4 semesters.",
        "gate_scholarship": "GATE-qualified students receive 50% tuition scholarship.",
        "attendance": "Students need 75% attendance for final exam eligibility.",
        "nlp": "NLP elective is coordinated by Dr. Kavitha Raman."
    }
    query_lower = query.lower()
    results = []
    for key, value in data.items():
        if key in query_lower or any(word in value.lower() for word in query_lower.split()):
            results.append(value)
    return "\n".join(results) if results else "No matching information found."

TOOLS = {
    "calculator": calculator,
    "search_college_info": search_college_info
}

def execute_tool(tool_name, args):
    if tool_name not in TOOLS:
        return f"Unknown tool: {tool_name}"
    return TOOLS[tool_name](**args)

## 2. ReAct Loop
ReAct = Reason -> Act -> Observe -> Repeat.

In [ ]:
def react_agent_for_fee_question(question, max_steps=5):
    memory = []

    for step in range(1, max_steps + 1):
        print(f"\nSTEP {step}")

        if step == 1:
            thought = "I need to find fee and scholarship information."
            action = {"tool": "search_college_info", "args": {"query": "mtech_fee gate_scholarship"}}

        elif step == 2:
            thought = "I have fee and scholarship. Now I need to calculate total after scholarship."
            action = {"tool": "calculator", "args": {"expression": "50000 * 4 * 0.5"}}

        else:
            thought = "I have enough information to answer."
            print("Reason:", thought)
            print("Final Answer: Total M.Tech tuition for a GATE scholar is Rs. 100000 over 4 semesters.")
            break

        print("Reason:", thought)
        print("Act:", action)

        observation = execute_tool(action["tool"], action["args"])
        print("Observe:", observation)

        memory.append({
            "step": step,
            "thought": thought,
            "action": action,
            "observation": observation
        })

    return memory

memory = react_agent_for_fee_question("GATE scholar - total M.Tech cost?")
memory

## 3. Max Iteration Guardrail

In [ ]:
def loop_risk_agent(max_steps=3):
    for step in range(1, max_steps + 1):
        print(f"Step {step}: Searching again...")
        time.sleep(0.3)

    print("\nStopped: maximum iteration limit reached.")
    print("Guardrail saved us from an infinite loop.")

loop_risk_agent(max_steps=3)

## 4. Tool Allowlist Guardrail

In [ ]:
ALLOWED_TOOLS = ["calculator", "search_college_info"]

def safe_execute_tool(tool_name, args):
    if tool_name not in ALLOWED_TOOLS:
        return f"Blocked: tool '{tool_name}' is not allowed."
    return execute_tool(tool_name, args)

print(safe_execute_tool("calculator", {"expression": "10+20"}))
print(safe_execute_tool("delete_database", {"table": "students"}))

## 5. Human Approval Guardrail

In [ ]:
RISKY_TOOLS = ["send_email", "delete_file", "update_database"]

def needs_approval(tool_name):
    return tool_name in RISKY_TOOLS

for tool in ["calculator", "send_email", "search_college_info", "delete_file"]:
    print(tool, "-> approval needed?", needs_approval(tool))

## 6. Logging Every Agent Step

In [ ]:
agent_logs = []

def logged_execute_tool(tool_name, args):
    result = safe_execute_tool(tool_name, args)
    log_entry = {
        "tool": tool_name,
        "args": args,
        "result": result
    }
    agent_logs.append(log_entry)
    return result

logged_execute_tool("calculator", {"expression": "100 + 200"})
logged_execute_tool("search_college_info", {"query": "attendance"})

pd.DataFrame(agent_logs)

## 7. Simple Cost Counter

In [ ]:
class CostCounter:
    def __init__(self):
        self.tool_calls = 0
        self.estimated_tokens = 0

    def add_tool_call(self, estimated_tokens=100):
        self.tool_calls += 1
        self.estimated_tokens += estimated_tokens

    def summary(self):
        return {
            "tool_calls": self.tool_calls,
            "estimated_tokens": self.estimated_tokens,
            "note": "Classroom estimate, not provider billing."
        }

counter = CostCounter()
counter.add_tool_call(120)
counter.add_tool_call(80)
counter.summary()

## Mini Exercise
Add one more guardrail: stop if the same tool is called with the same arguments twice.